In [ ]:
import numpy as np
import pandas as pd


In [ ]:
#Load the dataset

uga= pd.read_stata("/content/uga2014 (1).dta")
print(uga.shape)
uga.head()


(3987, 213)


,pid,sex,age,c4,c5,c6,c6other,c7,c8,c9,...,b6,b8,b9,b10,wstep1,wstep2,wstep3,psu,stratum,agerange
0,102001066,Men,23,15.0,University/ Higher institutions completed,Banyankore,,Never married,Student,4.0,...,No,4.28,1.13,No,4791.583008,4991.745117,5484.733887,1,urban,18-29
1,102001089,Women,20,10.0,O-Level completed,Baganda,.,Currently married,Unemployed (able to work),2.0,...,No,3.55,1.01,No,3468.302002,3497.881104,3698.048096,1,urban,18-29
2,102001133,Women,38,0.0,No formal schooling,Batoro,.,Separated,Self-employed,1.0,...,No,3.57,1.30,No,1758.876953,NaN,1880.235962,20,urban,30-49
3,102001178,Women,25,10.0,O-Level completed,Batoro,.,Currently married,Unemployed (able to work),3.0,...,No,3.16,0.52,No,3716.676025,3848.406006,3973.117920,1,urban,18-29
4,102001200,Women,32,13.0,Post graduate degree,Iteso,.,Never married,Non-government employee,1.0,...,No,4.44,1.50,No,1238.891968,1282.802002,1324.373047,1,urban,30-49


In [ ]:
#Checking for how the variables are stored

vars_to_check =[
    "sex", "age", "b1", "b5", "b6", "m7", "m8", "t1", "a5", "m11", "m12", "m4a", "m4b", "m5a", "m6a"
]

for col in vars_to_check:
    if col in uga.columns:
      print(f"\n--- {col}---")
      print(uga[col].value_counts(dropna=False).head(20))



--- sex---
sex
Women    2383
Men      1604
Name: count, dtype: int64

--- age---
age
18    186
30    155
25    153
27    143
24    141
22    137
35    136
26    136
23    134
28    126
40    121
20    121
19    121
32    116
29    113
21    105
31    103
37     99
38     96
34     95
Name: count, dtype: int64

--- b1---
b1
No     3694
NaN     269
Yes      24
Name: count, dtype: int64

--- b5---
b5
NaN    296
3.2    183
4.0    168
3.5    165
3.3    161
3.7    156
4.1    154
3.1    150
3.9    149
4.2    144
3.6    143
3.8    140
4.3    139
3.4    138
4.4    134
3.0    126
4.6    109
2.9    103
4.5     94
4.7     88
Name: count, dtype: int64

--- b6---
b6
No     3681
NaN     296
Yes      10
Name: count, dtype: int64

--- m7---
m7
No      3822
NaN       83
Yes       80
77.0       2
Name: count, dtype: int64

--- m8---
m8
No     2132
NaN    1649
Yes     206
Name: count, dtype: int64

--- t1---
t1
No     3638
Yes     345
NaN       4
Name: count, dtype: int64

--- a5---
a5
NaN    2600
Yes   

This shows that Uganda.dta uses labels and not numeric codes so we should recode using text labels and not 1 or 2.

In [ ]:
uga_sub =uga.copy()


In [ ]:
#Define the Uganda diabetes outcome
uga_sub["diabetes_binary"] =np.where(
    (uga_sub["b5"] >=7.0) | (uga_sub["b6"] =="Yes"),
    1,
    np.where(
        (uga_sub["b5"]. notna()) | (uga_sub["b6"] =="No"),
      0,
      np.nan
    )
)

uga_sub["diabetes_binary"].value_counts(dropna=False)


,count
diabetes_binary,
0.0,3645
NaN,296
1.0,46


In [ ]:
#Recode sex and age
uga_sub["sex_male"] =uga_sub["sex"].map({"Men":1, "Women":0})
uga_sub["age"]= uga_sub["age"]

print(uga_sub["sex_male"].value_counts(dropna=False))
print(uga_sub["age"].describe())

sex_male
0    2383
1    1604
Name: count, dtype: int64
count    3987.000000
mean       35.392024
std        13.037618
min        18.000000
25%        25.000000
50%        33.000000
75%        44.000000
max        69.000000
Name: age, dtype: float64


In [ ]:
#exclude currently pregnant respondents before BMI, becuase pregnancy can distort BMI and some metabolic interpretation

uga_sub =uga_sub[(uga_sub["m8"].isna()) | (uga_sub["m8"] !="Yes")].copy()
print(uga_sub.shape)


(3781, 215)


In [ ]:
#Derive BMI from height and weight

uga_sub["height_m"] =uga_sub["m11"] /100.0
uga_sub["bmi"] =uga_sub["m12"] / (uga_sub["height_m"] ** 2)

uga_sub.loc[(uga_sub["bmi"] <10) | (uga_sub["bmi"] >70), "bmi"] =np.nan

uga_sub["bmi"].describe()

,bmi
count,3680.000000
mean,22.928855
std,4.527040
min,11.251125
25%,20.080858
50%,22.074020
75%,24.679023
max,66.445183


In [ ]:
#Derive hypertension
uga_sub["sbp_mean"] =uga_sub[["m4a", "m5a", "m6a"]].mean(axis=1)
uga_sub["dbp_mean"] =uga_sub[["m4b", "m5b", "m6b"]].mean(axis=1)

uga_sub["hypertension"] =np.where(
    (uga_sub["sbp_mean"] >=140) | (uga_sub["dbp_mean"] >=90) |(uga_sub["m7"]=="Yes"),
    1,
    np.where(
        (uga_sub["sbp_mean"].notna()) |(uga_sub["dbp_mean"].notna()) |(uga_sub["m7"] =="No"),
        0,
        np.nan
    )
)
uga_sub["hypertension"].value_counts(dropna=False)


,count
hypertension,
0.0,2639
1.0,1061
NaN,81


In [ ]:
#Recode smoking

uga_sub["current_smoker"] =uga_sub["t1"].map({"Yes":1, "No":0})
uga_sub["current_smoker"].value_counts(dropna=False)


,count
current_smoker,
0,3432
1,345
NaN,4


In [ ]:
#Recode alcohol
uga_sub["current_alcohol"] =uga_sub["a5"].map({"Yes":1,"No":0})
uga_sub["current_alcohol"].value_counts(dropna=False)


,count
current_alcohol,
NaN,2442
1,1059
0,280


In [ ]:
def hm_to_minutes(h, m):
  h =pd.to_numeric(h, errors="coerce").fillna(0)
  m =pd.to_numeric(m, errors="coerce").fillna(0)
  return h* 60 + m

In [ ]:
for col in ["p3a","p3b", "p6a", "p6b", "p9a", "p9b", "p12a", "p12b", "p15a", "p15b"]:
    if col not in uga_sub.columns:
       uga_sub[col] =np.nan

work_vig =pd.to_numeric(uga_sub["p2"], errors='coerce').fillna(0) *hm_to_minutes(uga_sub["p3a"], uga_sub["p3b"])
work_mod =pd.to_numeric(uga_sub["p5"], errors='coerce').fillna(0) *hm_to_minutes(uga_sub["p6a"], uga_sub["p6b"])
travel_mod =pd.to_numeric(uga_sub["p8"], errors='coerce').fillna(0) *hm_to_minutes(uga_sub["p9a"], uga_sub["p9b"])
rec_vig =pd.to_numeric(uga_sub["p11"], errors='coerce').fillna(0) *hm_to_minutes(uga_sub["p12a"], uga_sub["p12b"])
rec_mod =pd.to_numeric(uga_sub["p14"], errors='coerce').fillna(0) *hm_to_minutes(uga_sub["p15a"], uga_sub["p15b"])

uga_sub["pa_minutes_week"] =(2 * work_vig) +work_mod +travel_mod + (2 * rec_vig) +rec_mod
uga_sub["physically_active"] =np.where(uga_sub["pa_minutes_week"] > 150, 1, 0)

uga_sub["physically_active"].value_counts(dropna=False)

,count
physically_active,
1,3554
0,227


In [ ]:
uganda_clean =uga_sub[[
    "diabetes_binary",
    "age",
    "sex_male",
    "bmi",
    "hypertension",
    "current_smoker",
    "current_alcohol",
    "physically_active"
]].copy()

print(uganda_clean.shape)
print("\nMissingness per variable:")
print(uganda_clean.isna().mean().sort_values())

uganda_clean.head()


(3781, 8)

Missingness per variable:
age                  0.000000
sex_male             0.000000
physically_active    0.000000
current_smoker       0.001058
hypertension         0.021423
bmi                  0.026713
diabetes_binary      0.074319
current_alcohol      0.645861
dtype: float64


,diabetes_binary,age,sex_male,bmi,hypertension,current_smoker,current_alcohol,physically_active
0,0.0,23,1,18.009218,0.0,0,NaN,1
1,0.0,20,0,21.449704,0.0,0,NaN,1
2,0.0,38,0,NaN,NaN,1,NaN,1
3,0.0,25,0,32.649730,0.0,0,1,1
4,0.0,32,0,21.057944,1.0,0,NaN,1


In [ ]:
uganda_clean.to_csv("/content/uganda_clean.csv", index=False)
print("Saved Uganda cleaned file to /content/uganda_clean.csv")


Saved Uganda cleaned file to /content/uganda_clean.csv


In [ ]:
uganda_clean.shape
uganda_clean["diabetes_binary"].value_counts(dropna=False)
uganda_clean.isna().mean().sort_values()


,0
age,0.000000
sex_male,0.000000
physically_active,0.000000
current_smoker,0.001058
hypertension,0.021423
bmi,0.026713
diabetes_binary,0.074319
current_alcohol,0.645861


In [ ]:
uganda_clean.to_csv("/content/uganda_clean.csv", index=False)
print("Saved Uganda cleaned file to /content/uganda_clean.csv")


Saved Uganda cleaned file to /content/uganda_clean.csv
